----
# Exploring Classification Models
----

### **Notebook Summary**
 
**Dataset Overview**

The dataset consists of six classes with the following distribution:

    operations: 230 examples
    legal: 212 examples
    finance: 208 examples
    personal: 185 examples
    spam: 184 examples
    unknown: 112 examples


In this notebook, I’ll experiment with three different classification models to understand how well they handle my dataset and what insights they can provide:

**1) Logistic Regression**

I’m starting with Logistic Regression because it’s simple, fast to train, and easy to interpret. It makes a solid baseline—if more complex models can’t outperform it, that’s often a sign that the data is limited or not informative enough.

**2) Support Vector Machine (SVM)**

Next, I’ll try an SVM classifier. SVMs are well-suited for high-dimensional data like embeddings. They focus on the hardest cases near the class boundaries (support vectors) and try to maximize the gap between classes, which makes them effective even with smaller datasets.

**3) Random Forest**

Finally, I’ll test a Random Forest. Unlike the previous models, Random Forest is an ensemble method that combines multiple decision trees. This allows it to capture more complex, non-linear relationships in the data that linear models might miss.

By comparing these models, I hope to get a better sense of how different approaches handle the task and what works best for this dataset.

## Set Up
----

In [19]:
import pandas as pd

import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## Load Embeddings
---

In [20]:
embeddings = joblib.load("labelled_embeddings.pkl")

In [21]:
labelled_df = pd.read_csv('data/labelled_emails.csv') # no index, no need for index_col

## Extract Labels
----

In [22]:
labelled_df = labelled_df[labelled_df['final_label'] != 'unknown']

In [23]:
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labelled_df['final_label'])

## Prepare Data for Modelling
---

In [24]:
X_train, X_val, y_train, y_val = train_test_split(
    embeddings,        
    labels_encoded,            
    test_size=0.2,          # 20% validation set
    random_state=14,  
    stratify=labels_encoded # ensure even split across train/test
)


## Training Models
---

In [25]:
classif_models = {
    'Logistic Regression': LogisticRegression(),
    'SVM': SVC(),
    'Random Forest': RandomForestClassifier(random_state=12)
}

In [26]:
results= []
for name, model in classif_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    results.append({
        'model': name,
        'precision': precision_score(y_val, pred, average='weighted'),
        'recall': recall_score(y_val, pred, average='weighted'),
        'f1-score': f1_score(y_val, pred, average='weighted'),
        'accuracy': accuracy_score(y_val, pred)
    })

    print(f"*********** Stored results for {name} ***********")


*********** Stored results for Logistic Regression ***********
*********** Stored results for SVM ***********
*********** Stored results for Random Forest ***********


In [27]:
results_df = pd.DataFrame(results)

In [28]:
results_df

,model,precision,recall,f1-score,accuracy
0,Logistic Regression,0.697386,0.686275,0.686425,0.686275
1,SVM,0.677060,0.666667,0.666961,0.666667
2,Random Forest,0.650110,0.627451,0.625134,0.627451


## Summary
----

Results from all three models are fairly similar. Since I used Logistic Regression as a baseline, the fact that the more complex models didn’t add much suggests that something else is causing poor performance. I suspect the main issue is the amount of labelled data, 1300 emails may not be enough for models to learn effectively especially something as nuanced as email text. I also only assigned a single label per email, but many emails could belong to multiple categories — this could be confusing the models.

I spent a long time labelling 1300 emails, and while it was a great learning experience, I don’t want to spend much more time labelling for this project. Next, I thought to try fine-tuning BERT on my labeled data, hoping it creates embeddings with a better understanding of the different categories.